In [1]:

import gradio as gr
import openai
import requests
import os
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import chromadb
from chromadb.config import Settings
import numpy as np
import pandas as pd
import datetime
import pytz  # pip install pytz if needed
import json


In [2]:
%load_ext dotenv
%dotenv ../05_src/.env
%dotenv ../05_src/.secrets

In [3]:
import os
from openai import OpenAI   
client = OpenAI()

In [4]:
# --- Guardrails ---
FORBIDDEN_TOPICS = ["cats", "dogs", "horoscopes", "zodiac", "taylor swift"]
# --- Memory ---
conversation_history = []
# simple dict for city coordinates
CITY_COORDS = {
    "toronto": (43.7, -79.42),
    "vancouver": (49.28, -123.12),
    "montreal": (45.50, -73.56),
    "calgary": (51.05, -114.07),
    "ottawa": (45.42, -75.69)
}



In [5]:

#--service 1
def extract_city(user_input):
    input_lower = user_input.lower()
    for city in CITY_COORDS.keys():
        if city in input_lower:
            return city
    return None

def get_weather_summary(city: str):
    city_lower = city.lower()
    if city_lower not in CITY_COORDS:
        return f"Sorry, I only know a few cities right now: {', '.join(CITY_COORDS.keys())}."

    lat, lon = CITY_COORDS[city_lower]
    url = (
        f"https://api.open-meteo.com/v1/forecast?"
        f"latitude={lat}&longitude={lon}&current_weather=true"
    )
    r = requests.get(url)
    if r.status_code != 200:
        return f"Couldn't fetch weather data for {city}."

    data = r.json().get("current_weather", {})
    temp = data.get("temperature")
    wind = data.get("windspeed")
    weather_info = f"Temperature {temp}°C and wind speed {wind} km/h in {city.title()}."

    # Prepare messages for chat completion
    messages = [
        {"role": "system", "content": "You are a friendly assistant giving short weather updates."},
        {"role": "user", "content": f"Summarize the following weather info for a friendly chat response:\n{weather_info}\nMake it concise, natural, and conversational."}
    ]

    completion = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=0.7
    )

    summary = completion.choices[0].message.content.strip()
    return summary



In [6]:
# --- test service 1
print(get_weather_summary("Toronto"))


Hey there! It’s a chilly 2.7°C in Toronto today, with a light breeze at 7.7 km/h. Perfect weather for a cozy sweater!


In [7]:

#-- service 2

# semantic_service_demo.py
# ---------------------------------------------------------------
# Demonstrates a semantic search service using Chroma + OpenAI.
# ---------------------------------------------------------------

from openai import OpenAI
import chromadb
from chromadb.config import Settings
import os


# Create a Chroma client (persistent on disk)

CHROMA_DIR = "chroma_semantic_store"
chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)

# ---------- Helper: get embeddings ----------
def get_embedding(text: str):
    """Use OpenAI embeddings for semantic indexing."""
    emb = client.embeddings.create(
        input=text,
        model="text-embedding-3-small"
    )
    return emb.data[0].embedding

# ---------- Initialize / Populate Chroma ----------
def init_collection():
    # Try to get if exists, else create new
    coll_names = [c.name for c in chroma_client.list_collections()]
    if "demo_docs" in coll_names:
        return chroma_client.get_collection("demo_docs")
    else:
        coll = chroma_client.create_collection(name="demo_docs")
        # add small sample docs
        docs = [
            "Photosynthesis is the process by which green plants convert sunlight into chemical energy.",
            "The capital of Canada is Ottawa, located in the province of Ontario.",
            "Python is a popular programming language known for its readability and versatility.",
            "The human heart has four chambers: two atria and two ventricles.",
            "Data engineering involves building and maintaining data pipelines and infrastructure."
        ]
        metadatas = [
            {"topic": "biology"},
            {"topic": "geography"},
            {"topic": "technology"},
            {"topic": "biology"},
            {"topic": "data"}
        ]
        ids = [f"id{i}" for i in range(len(docs))]
        embeddings = [get_embedding(d) for d in docs]
        coll.add(documents=docs, metadatas=metadatas, ids=ids, embeddings=embeddings)
        chroma_client.persist()
        return coll

collection = init_collection()

# ---------- Semantic search ----------
def semantic_query(question: str, top_k: int = 3):
    """Performs semantic search & uses OpenAI to summarize."""
    query_emb = get_embedding(question)
    results = collection.query(query_embeddings=[query_emb], n_results=top_k)
    
    docs = results["documents"][0]
    metas = results["metadatas"][0]

    # Prepare readable list
    joined = "\n".join([f"- ({m['topic']}) {d}" for d, m in zip(docs, metas)])
    
    # Ask the LLM to summarize / answer the question
    prompt = (
        f"You are a helpful assistant. Based on the following retrieved documents, "
        f"answer the user's question.\n\nDocuments:\n{joined}\n\n"
        f"Question: {question}\n\n"
        f"Please provide a concise, clear answer."
    )

    answer = client.chat.completions.create(
        model="gpt-4o-mini",  # or "gpt-3.5-turbo"
        messages=[
            {"role": "system", "content": "You summarize and answer questions based on context."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.5
    )

    return answer.choices[0].message.content.strip()



In [8]:
#--service 2 test 
q = "How do plants make their food?"
print(f"User Question: {q}\n")
response = semantic_query(q)
print("Assistant Answer:")
print(response)


User Question: How do plants make their food?

Assistant Answer:
Plants make their food through a process called photosynthesis. During photosynthesis, plants use sunlight, carbon dioxide from the air, and water from the soil to produce glucose (a type of sugar) and oxygen. The chlorophyll in the plant's leaves captures sunlight, which powers the chemical reactions that convert carbon dioxide and water into glucose.


In [9]:
# --- Service 3: Function Calling - Time info service ---
def get_time(city):
    city_lower = city.lower()
    tz_map = {
        "toronto": "America/Toronto",
        "new york": "America/New_York",
        "london": "Europe/London",
        "paris": "Europe/Paris",
    }
    if city_lower not in tz_map:
        return f"Sorry, I don't have timezone info for {city}."
    tz = pytz.timezone(tz_map[city_lower])
    now = datetime.datetime.now(tz)
    return f"The current time in {city.title()} is {now.strftime('%H:%M:%S')}."

functions = [
    {
        "name": "get_current_time",
        "description": "Get the current local time for a given city",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "The city to get the local time for"
                }
            },
            "required": ["city"]
        }
    }
]











In [ ]:
def chatbot(user_input, history=[]):
    # Compose messages with conversation history
    messages = [{"role": "system", "content": "You are a helpful assistant."}]
    for usr, bot in history:
        messages.append({"role": "user", "content": usr})
        messages.append({"role": "assistant", "content": bot})
    messages.append({"role": "user", "content": user_input})

    # Guardrails
    if any(topic in user_input.lower() for topic in FORBIDDEN_TOPICS):
        refusal = "Sorry, I can't discuss that topic. Please ask something else."
        history.append((user_input, refusal))
        return refusal, history

    # Simple routing: keyword based for service1, service2 or call function
    if any(k in user_input.lower() for k in ["weather", "temperature", "wind", "forecast"]):
        city = extract_city(user_input)
        if city:
            reply = get_weather_summary(city)
            history.append((user_input, reply))
            return reply, history

    elif any(k in user_input.lower() for k in ["search", "find", "tell me about", "information"]):
        reply = semantic_query(user_input)
        history.append((user_input, reply))
        return reply, history

    else:
        # OpenAI function-calling usage
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            functions=functions,
            function_call="auto",
            temperature=0.7,
        )
        message = response.choices[0].message

        if hasattr(message, "function_call") and message.function_call is not None:
            func_name = message.function_call.name
            arguments = json.loads(message.function_call.arguments)
            print(f"Calling function: {func_name} with args {arguments}")  # debug print

            if func_name == "get_current_time":
                func_response = get_time(arguments.get("city", ""))
                if ("Sorry, I don't have timezone info" in func_response):
                     history.append((user_input, "I can not answer the time about this city"))
                     return "I can not answer the time about this city", history

               
            else:
                func_response = "Function not implemented."

            messages.append(message)
            messages.append({
                "role": "function",
                "name": func_name,
                "content": func_response,
            })

            second_response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=messages,
                temperature=0.7,
            )
            reply = second_response.choices[0].message.content
            history.append((user_input, reply))
            return reply, history

        # No function call detected and no other service matched
        fallback_reply = "I can not answer that."
        history.append((user_input, fallback_reply))
        return fallback_reply, history


# --- Gradio interface ---
with gr.Blocks() as demo:
    chatbot_ui = gr.Chatbot()
    user_input = gr.Textbox(placeholder="Ask me about weather, info, or time", label="Your message")
    submit_btn = gr.Button("Send")
    
    def respond(msg, history):
        response, updated_history = chatbot(msg, history or [])
        return "", updated_history
    
    submit_btn.click(respond, inputs=[user_input, chatbot_ui], outputs=[user_input, chatbot_ui])

demo.launch()

D:\Temp\ipykernel_16600\195131872.py:64: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot_ui = gr.Chatbot()


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Calling function: get_current_time with args {'city': 'Toronto'}
